# Read Curriculum Data from S3

This notebook reads the processed curriculum data from S3 using AWS credentials from `env.json`.

In [21]:
# Install dependencies if needed
! pip install boto3 pandas pyarrow s3fs

In [4]:
import os
import json
import pandas as pd
import s3fs

## 1. Load AWS Credentials from env.json

In [6]:
# Load credentials from env.json
with open('env.json', 'r') as f:
    env = json.load(f)

AWS_ACCESS_KEY_ID = env['AWS_ACCESS_KEY_ID']
AWS_SECRET_ACCESS_KEY = env['AWS_SECRET_ACCESS_KEY']
AWS_REGION = env.get('AWS_REGION', 'us-east-1')

# Set environment variables
os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
os.environ['AWS_DEFAULT_REGION'] = AWS_REGION

## 2. S3 Path Configuration

In [2]:
# Base path for curriculum data
S3_BASE_PATH = "s3://t1-dataacquisition-datasets/processed_dataset/curriculum_data/"

# Specific paths
source = "books"  # Change as needed 'books', 'ncert', 'sangraha_te'
BANDS_PATH = f"{S3_BASE_PATH}source={source}/bands/"
REJECTIONS_PATH = f"{S3_BASE_PATH}source={source}/rejections/"

print(f"Bands path: {BANDS_PATH}")
print(f"Rejections path: {REJECTIONS_PATH}")

Bands path: s3://t1-dataacquisition-datasets/processed_dataset/curriculum_data/source=books/bands/
Rejections path: s3://t1-dataacquisition-datasets/processed_dataset/curriculum_data/source=books/rejections/


## 3. List Available Data

In [7]:
# Create S3 filesystem
fs = s3fs.S3FileSystem()

# List sources available
base_path = "t1-dataacquisition-datasets/processed_dataset/curriculum_data/"
try:
    sources = fs.ls(base_path)
    print("Available sources:")
    for src in sources:
        print(f"  - {src}")
except Exception as e:
    print(f"Error listing sources: {e}")

Available sources:
  - t1-dataacquisition-datasets/processed_dataset/curriculum_data/source=books
  - t1-dataacquisition-datasets/processed_dataset/curriculum_data/source=ncert
  - t1-dataacquisition-datasets/processed_dataset/curriculum_data/source=sangraha_te


In [6]:
# List bands within a source
bands_base = f"t1-dataacquisition-datasets/processed_dataset/curriculum_data/source={source}/bands/"

try:
    bands = fs.ls(bands_base)
    print(f"Available bands for {source}:")
    for band in bands:
        print(f"  - {band.split('/')[-1]}")
except Exception as e:
    print(f"Error listing bands: {e}")

Available bands for books:
  - band=B0
  - band=B1
  - band=B2
  - band=B3
  - band=B4


## 4. Read Data with Pandas

In [7]:
# Read a specific band (e.g., B3 - intermediate difficulty)
band = "B1"
band_path = f"s3://t1-dataacquisition-datasets/processed_dataset/curriculum_data/source={source}/bands/band={band}/"

print(f"Reading from: {band_path}")

# Read with pandas
df = pd.read_parquet(
    band_path,
    storage_options={
        'key': AWS_ACCESS_KEY_ID,
        'secret': AWS_SECRET_ACCESS_KEY
    }
)

print(f"Loaded {len(df):,} rows")
df.head()

Reading from: s3://t1-dataacquisition-datasets/processed_dataset/curriculum_data/source=books/bands/band=B1/


/Users/pankajkumar/Documents/git/TSAI/ERA4/capstone-antigravity/.venv/lib/python3.13/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


Loaded 7,386 rows


,uuid,id,file_path,source,domain,text,hash,language,metadata,band_p_B0,...,reasoning_score,code_score,math_score,table_score,byte_length,word_count,unique_token_ratio,compression_ratio,token_count_estimate,fertility_estimate
0,217ed753-4f59-4862-9299-aa20399b0cec,eb9421a1271444b481e622456448d88bdcd20e24,/source=books/part-00004-b476a77d-9ec8-48a2-bd...,books,literature,"Transcriber's notes:\n\n 1. The disease ""c...",be2047b95adabd29d26825cd8ff4641531269388a59607...,en,"{""gutenberg_metadata_available"":true,""issued_o...",0.0,...,0,9,2,0,224089,43486,0.124799,1.0,56531,3.964002
1,de7ebad9-6363-410c-a80e-ac6f075ce925,dbbcebc51df25739cc46ece56b9890640579e6e8,/source=books/part-00004-b476a77d-9ec8-48a2-bd...,books,literature,[Illustration: Cover art]\n\n\n\n[Frontispiece...,2074075044b12f0d2e403baef5fd7db315f1ba56977b12...,en,"{""gutenberg_metadata_available"":true,""issued_o...",0.0,...,0,9,4,0,75196,12831,0.320552,1.0,16680,4.508153
2,bb91443d-1686-4650-a8eb-9b02c576eef8,6acce25fb7e221b583c8f12fad1f88e9210b788b,/source=books/part-00004-b476a77d-9ec8-48a2-bd...,books,literature,provided by the Internet Archive\n\n\n\nTHE OL...,ce34b8dff2169f0920c37ad45e983bd3faaffc88b7bb20...,en,"{""gutenberg_metadata_available"":true,""issued_o...",0.0,...,0,3,2,0,123177,22932,0.216074,1.0,29811,4.131931
3,1ad6111d-3044-406d-b3f7-2e596b5a28d4,bf0986644c1e49f89a98c56cc0c0d5b1b6e85a71,/source=books/part-00004-b476a77d-9ec8-48a2-bd...,books,literature,* * * * *\n\n +----...,6e924e0c9d498112ca37d0caa95237b9596e9a9827e7ab...,en,"{""gutenberg_metadata_available"":true,""issued_o...",0.0,...,0,9,4,0,73670,13626,0.239689,1.0,17713,4.159092
4,3ab7f566-c308-4b60-921d-1109a0924190,2e0ed6b76b507ae3d16663eb83726fb68ce70552,/source=books/part-00004-b476a77d-9ec8-48a2-bd...,books,literature,EVERYDAY LIFE LIBRARY No. 3\nPublished by EVER...,dc5c492e3a89f64e51e344b0311446a353d92e9420c6c1...,en,"{""gutenberg_metadata_available"":true,""issued_o...",0.0,...,0,3,4,0,121187,22911,0.197940,1.0,29784,4.068862


In [18]:
df[['uuid','band_p_B0', 'band_p_B1', 'band_p_B2', 'band_p_B3']]

,uuid,band_p_B0,band_p_B1,band_p_B2,band_p_B3
0,217ed753-4f59-4862-9299-aa20399b0cec,0.000000,0.346154,0.615385,0.038462
1,de7ebad9-6363-410c-a80e-ac6f075ce925,0.000000,0.333333,0.592593,0.074074
2,bb91443d-1686-4650-a8eb-9b02c576eef8,0.000000,0.600000,0.400000,0.000000
3,1ad6111d-3044-406d-b3f7-2e596b5a28d4,0.000000,0.291246,0.634680,0.074074
4,3ab7f566-c308-4b60-921d-1109a0924190,0.000000,0.230769,0.730769,0.038462
...,...,...,...,...,...
7381,61f72b55-be31-4f45-b6cb-172042a89363,0.000000,0.346154,0.615385,0.038462
7382,161bc8fe-ddc6-4bc4-ab6e-5f1ed14d8e4f,0.014509,0.562004,0.350488,0.072999
7383,2d6964c1-c20a-46ee-a8dc-e194c9546254,0.000000,0.576923,0.384615,0.038462
7384,ec714d0d-3045-4c7e-8a23-75f1b76ed049,0.000000,0.238077,0.723462,0.038462


In [20]:
df.head(100).to_csv('filename1.csv', index=False, encoding='utf-8')

In [17]:
# View columns and dtypes
print("Columns:")
print(df.dtypes)

Columns:
uuid                        str
id                          str
file_path                   str
source                      str
domain                      str
text                        str
hash                        str
language                    str
metadata                    str
band_p_B0               float64
band_p_B1               float64
band_p_B2               float64
band_p_B3               float64
band_p_B4               float64
band_p_B5               float64
difficulty_score        float64
agentic_score             int32
cot_score                 int32
reasoning_score           int32
code_score                int32
math_score                int32
table_score               int32
byte_length               int32
word_count                int32
unique_token_ratio      float64
compression_ratio       float64
token_count_estimate      int32
fertility_estimate      float64
dtype: object


In [9]:
# Sample a few rows to inspect text content
sample = df.sample(n=min(5, len(df)))
for idx, row in sample.iterrows():
    print(f"\n{'='*60}")
    print(f"UUID: {row.get('uuid', 'N/A')}")
    print(f"Difficulty Score: {row.get('difficulty_score', 'N/A'):.3f}")
    print(f"Text Preview: {row.get('text', '')[:300]}...")


UUID: 838daff2-0d11-4d11-89f2-9ad99fbce806
Difficulty Score: 0.340
Text Preview: Malcolm Farmer and the Online Distributed Proofreading
Team at http://www.pgdp.net





PUNCH, OR THE LONDON CHARIVARI.

VOL. 109.

OCTOBER 12, 1895.


[Illustration: TRUE LIBERALITY.

_Old Millionaire._ "GEORGE, I'VE JUST SENT A GUINEA TO THE 'BALACLAVA
VETERANS' HOME.'"

_His only Son and Heir._ "...

UUID: 2319fd4f-8283-49c9-a336-eb30140da59e
Difficulty Score: 0.324
Text Preview: generously made available by The Internet Archive/American
Libraries.)




  +------------------------------------------------------------+
  | Transcriber's Note:                                        |
  |                                                            |
  | Obvious typographical erro...

UUID: 1ede3557-f6ff-45d2-a413-f7828bf282ef
Difficulty Score: 0.260
Text Preview: HOW MEMBERS OF CONGRESS ARE BRIBED.

An Open Letter.

A Protest and a Petition.

From a Citizen of California to the United States Congress

by Jo

In [15]:
! pip install matplotlib

In [16]:
# Visualize band distribution
import matplotlib.pyplot as plt

if band_counts:
    plt.figure(figsize=(10, 5))
    plt.bar(band_counts.keys(), band_counts.values(), color='steelblue')
    plt.xlabel('Curriculum Band')
    plt.ylabel('Document Count')
    plt.title(f'Document Distribution by Band ({source})')
    plt.tight_layout()
    plt.show()

NameError: name 'band_counts' is not defined

In [ ]:
from curriculum_validator import CurriculumValidator

In [ ]:
validator = CurriculumValidator(df)
results = validator.validate_all()

CURRICULUM DATA VALIDATION REPORT

📋 SCHEMA VALIDATION
----------------------------------------
✓ All required columns present (17)

🔍 DATA INTEGRITY
----------------------------------------
  uuid: 0 nulls ✓
  text: 0 nulls ✓
  hash: 0 nulls ✓
  difficulty_score: 0 nulls ✓
  Duplicate UUIDs: 0 ✓
  Duplicate hashes: 4 ⚠️ (exact dupes)

📏 TOKEN LENGTH VALIDATION
----------------------------------------


KeyError: 'band'

In [ ]:
validator.spot_check(n=3, band='band_p_B0')


🔬 SPOT CHECK - Sample Documents
----------------------------------------


KeyError: 'band'

In [ ]:
validator.spot_check(n=3, band='B0')

In [8]:
source = "books"
rej_lvl_1_path = f"s3://t1-dataacquisition-datasets/processed_dataset/curriculum_data/source={source}/rejections/rejection_level=1/"

print(f"Reading from: {rej_lvl_1_path}")

# Read with pandas
df = pd.read_parquet(
    rej_lvl_1_path,
    storage_options={
        'key': AWS_ACCESS_KEY_ID,
        'secret': AWS_SECRET_ACCESS_KEY
    }
)

print(f"Loaded {len(df):,} rows")
df.head()

Reading from: s3://t1-dataacquisition-datasets/processed_dataset/curriculum_data/source=books/rejections/rejection_level=1/


/Users/pankajkumar/Documents/git/TSAI/ERA4/capstone-antigravity/.venv/lib/python3.13/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


Loaded 4 rows


,uuid,id,file_path,source,domain,text,hash,language,metadata,byte_length,word_count,token_count_estimate,fertility_estimate,is_rejected,rejection_reason,unique_token_ratio,compression_ratio
0,89e3f271-bf21-4511-b41f-10c8b1ac11f3,NaN,/source=books/part-00006-b476a77d-9ec8-48a2-bd...,books,literature,NaN,NaN,en,NaN,NaN,-2,-2,1.0,True,too_short_tokens,NaN,NaN
1,274009bd-f997-43fd-a784-bed3d2006db7,NaN,/source=books/part-00000-b476a77d-9ec8-48a2-bd...,books,literature,NaN,NaN,en,NaN,NaN,-2,-2,1.0,True,too_short_tokens,NaN,NaN
2,7d5685f0-b9a0-4b65-9e66-73fe839161eb,NaN,/source=books/part-00031-b476a77d-9ec8-48a2-bd...,books,literature,NaN,NaN,en,NaN,NaN,-2,-2,1.0,True,too_short_tokens,NaN,NaN
3,4c4f83d4-d29c-4f91-b817-c6b910c0986c,NaN,/source=books/part-00030-b476a77d-9ec8-48a2-bd...,books,literature,NaN,NaN,en,NaN,NaN,-2,-2,1.0,True,too_short_tokens,NaN,NaN


In [9]:
df.head(100).to_csv('rej_lvl1_filename.csv', index=False, encoding='utf-8')

In [10]:
source = "sangraha_te"
rej_lvl_2_path = f"s3://t1-dataacquisition-datasets/processed_dataset/curriculum_data/source={source}/rejections/rejection_level=2/"

print(f"Reading from: {rej_lvl_2_path}")

# Read with pandas
df = pd.read_parquet(
    rej_lvl_2_path,
    storage_options={
        'key': AWS_ACCESS_KEY_ID,
        'secret': AWS_SECRET_ACCESS_KEY
    }
)

print(f"Loaded {len(df):,} rows")
df.head()

Reading from: s3://t1-dataacquisition-datasets/processed_dataset/curriculum_data/source=sangraha_te/rejections/rejection_level=2/
Loaded 9 rows


,uuid,id,file_path,source,domain,text,hash,language,metadata,byte_length,word_count,token_count_estimate,fertility_estimate,is_rejected,rejection_reason,unique_token_ratio,compression_ratio
0,b8e41a21-5f78-47e8-906d-d1575e378b6e,8b2c5963c0f53537ddb07016c4cfa5fc,/source=sangraha_te/part-00059-e317e1fd-cd91-4...,sangraha_te,web,"అంకగణితం,అంకెలు-సంఖ్యలు,అంగారకుడు,అంటార్కిటికా...",5b7ea3611ee8ed001f51f58e7c3074bfc4b9dc80452a86...,te,NaN,21984,380,494,16.202429,True,excessive_whitespace,0.989474,2.746627
1,a82bced0-d7c7-4913-a520-e485cf1beb65,2217b5aa7ad55fd1ce92493983a3b1a7b7999082,/source=sangraha_te/part-00112-e317e1fd-cd91-4...,sangraha_te,web,పూణేకి చెందిన ఓ యువతి ఫొటో షూట్ను అందరిలా కాకు...,162fd2f408bf01e2b22bfce538dc4d5be40a4a3b0b5287...,te,NaN,1841,134,174,5.327586,True,orphaned_thread_fragment,0.858209,1.985976
2,b18bb11f-385d-4c16-a065-cd29de83775b,5d386512af0f063c05ca6ee644344088be413098,/source=sangraha_te/part-00113-e317e1fd-cd91-4...,sangraha_te,web,Shakun Shastra : పని మీద ఊరెళ్తున్నారా...అయితే...,2cf91bb4c137b2e790fc6935214aed252fa8e8b0ab46eb...,te,NaN,2725,148,192,5.526042,True,orphaned_thread_fragment,0.797297,2.568332
3,04b64e21-8daf-4827-8fb3-8454fb26fcae,495a15d8eacf80db4c0308ed184a66060858f4fe,/source=sangraha_te/part-00121-e317e1fd-cd91-4...,sangraha_te,web,">> జ్యోతిష్య శాస్త్రం ప్రకారం, మేషరాశిపై గణేశు...",9349085c8b2520575bd070dfde213a6776ff079a41a238...,te,NaN,2433,135,175,5.445714,True,orphaned_thread_fragment,0.533333,2.552991
4,eedb63be-f07d-4ea9-9cbe-89a8d857ede0,91e53ddce8028fa0c85033824fd9446c8535bed4,/source=sangraha_te/part-00049-e317e1fd-cd91-4...,sangraha_te,web,Bitter Gourd (కాకర కాయ) కూర తినడం చాలా మందికి ...,0f955510eef62a72da16b68313a94c1f9ff305c8c659d4...,te,NaN,3164,148,192,6.229167,True,orphaned_thread_fragment,0.790541,2.645485


In [11]:
df.head(100).to_csv('rej_lvl2_filename.csv', index=False, encoding='utf-8')

In [13]:
import json
import pandas as pd

# Option 1: Using pandas (recommended for tabular data)
df = pd.read_json("/Users/pankajkumar/Downloads/b1_all_sources_dedup_collapsed1.jsonl", lines=True)
df.head()

# # Option 2: Using standard json (for more control)
# data = []
# with open("path/to/your/file.jsonl", "r") as f:
#     for line in f:
#         data.append(json.loads(line))

,uuid,id,text,band_p_B1,word_count,token_count_estimate,source
0,217ed753-4f59-4862-9299-aa20399b0cec,eb9421a1271444b481e622456448d88bdcd20e24,"Transcriber's notes:\n\n 1. The disease ""c...",0.346154,43486,56531,books
1,de7ebad9-6363-410c-a80e-ac6f075ce925,dbbcebc51df25739cc46ece56b9890640579e6e8,[Illustration: Cover art]\n\n\n\n[Frontispiece...,0.333333,12831,16680,books
2,bb91443d-1686-4650-a8eb-9b02c576eef8,6acce25fb7e221b583c8f12fad1f88e9210b788b,provided by the Internet Archive\n\n\n\nTHE OL...,0.600000,22932,29811,books
3,1ad6111d-3044-406d-b3f7-2e596b5a28d4,bf0986644c1e49f89a98c56cc0c0d5b1b6e85a71,* * * * *\n\n +----...,0.291246,13626,17713,books
4,3ab7f566-c308-4b60-921d-1109a0924190,2e0ed6b76b507ae3d16663eb83726fb68ce70552,EVERYDAY LIFE LIBRARY No. 3\nPublished by EVER...,0.230769,22911,29784,books


In [15]:
df.head(3).to_csv('b1_all_sources_dedup_collapsed.csv', index=False, encoding='utf-8')